In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Main Data + Dummy.xlsx to Main Data + Dummy.xlsx


In [3]:
df = pd.read_excel('Main Data + Dummy.xlsx')

In [4]:
!pip install statsmodels

In [5]:
import statsmodels.api as sm

# Filter to top 9 domains, exclude zero basket
top9 = ['amazon.com','kohls.com','walmart.com','ebay.com','gap.com',
        'macys.com','blair.com','target.com','qvc.com']
df_model = df[df['domain_name'].isin(top9)].copy()
df_model = df_model[df_model['basket_tot'] > 0].copy()

# Winsorize at 99th percentile
p99 = df_model['basket_tot'].quantile(0.99)
df_model['basket_tot_wins'] = df_model['basket_tot'].clip(upper=p99)

# Create log dependent variable
df_model['log_basket'] = np.log(df_model['basket_tot_wins'] + 1)

# Create domain dummies (amazon.com is reference, not included)
domains = ['kohls.com','walmart.com','ebay.com','gap.com',
           'macys.com','blair.com','target.com','qvc.com']
for d in domains:
    col = 'D_' + d.replace('.','_')
    df_model[col] = (df_model['domain_name'] == d).astype(int)

# Create quarter dummies (Q1 is reference)
for q in [2, 3, 4]:
    df_model[f'Q{q}'] = (df_model['Quarter'] == q).astype(int)

# Create region dummies (Northeast is reference)
for r in [2, 3, 4]:
    df_model[f'Region_{r}'] = (df_model['census_region Code'] == r).astype(int)

# All independent variables
domain_vars  = ['D_' + d.replace('.','_') for d in domains]
quarter_vars = ['Q2','Q3','Q4']
region_vars  = ['Region_2','Region_3','Region_4']
hh_size_vars = ['HH_Size_2','HH_Size_3','HH_Size_4','HH_Size_5','HH_Size_6plus']
income_vars  = ['Income_25_39k','Income_40_59k','Income_60_74k','Income_75_99k',
                'Income_100_149k','Income_150_199k','Income_200k_plus']
age_vars     = ['Gen_Z_YoungMillennial','Millennial','GenX']
edu_vars     = ['Edu_Some_College','Edu_Associate','Edu_Bachelors',
                'Edu_Graduate','Edu_Missing']
other_vars   = ['children Code','country_of_origin Code']

all_vars = (domain_vars + quarter_vars + region_vars + hh_size_vars +
            income_vars + age_vars + edu_vars + other_vars)

# Build X and y
X = df_model[all_vars].copy()
X = sm.add_constant(X)
y = df_model['log_basket']

# Drop rows with missing values
clean = X.notna().all(axis=1) & y.notna()
X = X[clean]
y = y[clean]
groups = df_model[clean]['machine_id']

# Run OLS with clustered standard errors
model  = sm.OLS(y, X)
result = model.fit(cov_type='cluster', cov_kwds={'groups': groups})

print(result.summary())
print(f"\nSample size:          {int(result.nobs):,}")
print(f"R-squared:            {result.rsquared:.4f}")
print(f"Adjusted R-squared:   {result.rsquared_adj:.4f}")
print(f"99th pct winsor cap:  ${p99:.2f}")

                            OLS Regression Results                            
Dep. Variable:             log_basket   R-squared:                       0.134
Model:                            OLS   Adj. R-squared:                  0.132
Method:                 Least Squares   F-statistic:                     13.15
Date:                Sun, 03 May 2026   Prob (F-statistic):           1.00e-74
Time:                        19:03:42   Log-Likelihood:                -29482.
No. Observations:               23312   AIC:                         5.904e+04
Df Residuals:                   23275   BIC:                         5.934e+04
Df Model:                          36                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      3

In [6]:
summary_df = pd.DataFrame({
    'Variable': result.params.index,
    'Coefficient': result.params.values.round(4),
    'Std_Error': result.bse.values.round(4),
    'z_stat': result.tvalues.values.round(3),
    'p_value': result.pvalues.values.round(4)
})

def stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    else: return ''

summary_df['Significance'] = summary_df['p_value'].apply(stars)
summary_df.to_csv('model1_results.csv', index=False)

from google.colab import files
files.download('model1_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import pandas as pd
import numpy as np

df = pd.read_excel('Main Data + Dummy.xlsx')

# Filter to top 10 domains
top10 = ['amazon.com','kohls.com','walmart.com','ebay.com','gap.com',
         'macys.com','blair.com','womanwithin.com','target.com','qvc.com']
df = df[df['domain_name'].isin(top10)].copy()

# ============================================================
# ANALYSIS 1: DOMAIN LOYALTY
# ============================================================

# Count transactions per machine_id per domain
loyalty = df.groupby(['machine_id','domain_name']).size().reset_index(name='txn_count')

# Total transactions per machine_id
total_txns = df.groupby('machine_id').size().reset_index(name='total_txns')

# Number of distinct domains per machine_id
distinct_domains = df.groupby('machine_id')['domain_name'].nunique().reset_index(name='distinct_domains')

# Most purchased domain per machine_id
top_domain = loyalty.loc[loyalty.groupby('machine_id')['txn_count'].idxmax()][['machine_id','domain_name','txn_count']]
top_domain.columns = ['machine_id','top_domain','top_domain_txns']

# Merge
loyalty_df = total_txns.merge(distinct_domains, on='machine_id').merge(top_domain, on='machine_id')

# Loyalty score = share of transactions on top domain
loyalty_df['loyalty_score'] = loyalty_df['top_domain_txns'] / loyalty_df['total_txns']

print("=== DOMAIN LOYALTY SUMMARY ===")
print(f"Total unique households: {len(loyalty_df):,}")
print(f"\nAverage distinct domains per household: {loyalty_df['distinct_domains'].mean():.2f}")
print(f"Average loyalty score: {loyalty_df['loyalty_score'].mean():.2%}")

print("\nLoyalty score distribution:")
print(loyalty_df['loyalty_score'].describe().round(3))

print("\nShare of households by loyalty score:")
bins = [0, 0.5, 0.75, 0.99, 1.01]
labels = ['Low (<50%)', 'Medium (50-75%)', 'High (75-99%)', 'Fully Loyal (100%)']
loyalty_df['loyalty_group'] = pd.cut(loyalty_df['loyalty_score'], bins=bins, labels=labels)
print(loyalty_df['loyalty_group'].value_counts().sort_index())

print("\nTop domain by loyalty group (most common):")
print(loyalty_df.groupby('loyalty_group')['top_domain'].value_counts().groupby(level=0).head(3))

# ============================================================
# ANALYSIS 2: LOYAL VS NON-LOYAL SPENDING
# ============================================================

# Merge loyalty back to transactions
df2 = df[df['prod_totprice'] > 0].merge(loyalty_df[['machine_id','loyalty_score','loyalty_group','distinct_domains']], on='machine_id')

print("\n=== SPENDING BY LOYALTY GROUP ===")
spending = df2.groupby('loyalty_group')['prod_totprice'].agg(['mean','median','count']).round(2)
print(spending)

# ============================================================
# ANALYSIS 3: SAME CATEGORY DIFFERENT DOMAIN (cross-domain buying)
# ============================================================

print("\n=== CROSS DOMAIN BUYING ===")
print("Households that bought from more than 1 domain:")
multi_domain = loyalty_df[loyalty_df['distinct_domains'] > 1]
print(f"  Count: {len(multi_domain):,} households ({len(multi_domain)/len(loyalty_df):.1%})")

print("\nDistribution of distinct domains per household:")
print(loyalty_df['distinct_domains'].value_counts().sort_index())

# ============================================================
# ANALYSIS 4: LOYALTY BY DOMAIN
# ============================================================

print("\n=== LOYALTY SCORE BY TOP DOMAIN ===")
domain_loyalty = loyalty_df.groupby('top_domain')['loyalty_score'].agg(['mean','median','count']).round(3)
domain_loyalty.columns = ['Avg Loyalty Score','Median Loyalty Score','# Households']
domain_loyalty = domain_loyalty.sort_values('Avg Loyalty Score', ascending=False)
print(domain_loyalty)

=== DOMAIN LOYALTY SUMMARY ===
Total unique households: 7,303

Average distinct domains per household: 1.22
Average loyalty score: 94.13%

Loyalty score distribution:
count    7303.000
mean        0.941
std         0.143
min         0.250
25%         1.000
50%         1.000
75%         1.000
max         1.000
Name: loyalty_score, dtype: float64

Share of households by loyalty score:
loyalty_group
Low (<50%)             351
Medium (50-75%)        595
High (75-99%)          294
Fully Loyal (100%)    6063
Name: count, dtype: int64

Top domain by loyalty group (most common):
loyalty_group       top_domain 
Low (<50%)          amazon.com      227
                    kohls.com        39
                    ebay.com         19
Medium (50-75%)     amazon.com      206
                    kohls.com        95
                    walmart.com      60
High (75-99%)       amazon.com      103
                    kohls.com        60
                    gap.com          26
Fully Loyal (100%)  amazon.com

/tmp/ipykernel_1273/759401743.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(loyalty_df.groupby('loyalty_group')['top_domain'].value_counts().groupby(level=0).head(3))
/tmp/ipykernel_1273/759401743.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(loyalty_df.groupby('loyalty_group')['top_domain'].value_counts().groupby(level=0).head(3))
/tmp/ipykernel_1273/759401743.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silen

In [8]:
print(df.shape)
print(df['prod_totprice'].describe())

(26098, 56)
count    26098.000000
mean        24.590647
std         34.203075
min          0.000000
25%         12.950000
50%         18.990000
75%         28.000000
max       2231.199997
Name: prod_totprice, dtype: float64


In [9]:
import pandas as pd
import numpy as np
import re
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# STEP 1: PREPARE DATA
# ─────────────────────────────────────────────
top10 = ['amazon.com','kohls.com','walmart.com','ebay.com','gap.com',
         'macys.com','blair.com','womanwithin.com','target.com','qvc.com']
df2 = df[df['domain_name'].isin(top10)].copy()
df2 = df2[df2['prod_totprice'] > 0].copy()
df2['event_date'] = pd.to_datetime(df2['event_date'])
df2['month'] = df2['event_date'].dt.month
df2['month_name'] = df2['event_date'].dt.strftime('%B')
print(f"Working sample: {len(df2):,} transactions")

# ─────────────────────────────────────────────
# STEP 2: LOYAL VS MULTI-PLATFORM
# ─────────────────────────────────────────────
domains_per_hh = df2.groupby('machine_id')['domain_name'].nunique().reset_index(name='n_domains')
domains_per_hh['customer_type'] = domains_per_hh['n_domains'].apply(
    lambda x: 'Loyal' if x == 1 else 'Multi-Platform')
df2 = df2.merge(domains_per_hh[['machine_id','customer_type']], on='machine_id')
print("\nHouseholds by type:")
print(df2.groupby('customer_type')['machine_id'].nunique())

# ─────────────────────────────────────────────
# STEP 3: CHANNEL TYPE
# ─────────────────────────────────────────────
dual = ['kohls.com','gap.com','macys.com','walmart.com',
        'target.com','womanwithin.com','blair.com']
df2['channel_type'] = df2['domain_name'].apply(
    lambda x: 'Dual Channel' if x in dual else 'Single Channel')

# ─────────────────────────────────────────────
# STEP 4: CLOTHING TYPE
# ─────────────────────────────────────────────
keywords = {
    'Dress':      r'\b(dress|gown|sundress|maxi dress|midi dress)\b',
    'Top':        r'\b(top|blouse|tank|cami|tunic|tee|t-shirt|shirt|polo)\b',
    'Pants':      r'\b(pant|pants|trouser|legging|jogger|capri)\b',
    'Jeans':      r'\b(jean|jeans|denim)\b',
    'Jacket':     r'\b(jacket|blazer|bomber|vest)\b',
    'Coat':       r'\b(coat|parka|peacoat|trench|overcoat)\b',
    'Sweater':    r'\b(sweater|sweatshirt|hoodie|pullover|cardigan)\b',
    'Skirt':      r'\b(skirt)\b',
    'Swimwear':   r'\b(swimsuit|bikini|bathing suit)\b',
    'Shoes':      r'\b(shoe|boot|sneaker|sandal|heel|flat|loafer)\b',
    'Activewear': r'\b(yoga|athletic|workout|sports bra|running)\b',
}

def classify(name):
    if pd.isna(name): return 'Other'
    n = str(name).lower()
    for cat, pat in keywords.items():
        if re.search(pat, n, re.IGNORECASE):
            return cat
    return 'Other'

df2['clothing_type'] = df2['prod_name'].apply(classify)
print("\nClothing type counts:")
print(df2['clothing_type'].value_counts())

# ─────────────────────────────────────────────
# STEP 5: BRANDED VS GENERIC
# ─────────────────────────────────────────────
brands = ['nike','adidas','levi','calvin klein','ralph lauren','tommy hilfiger',
          'michael kors','kate spade','coach','gap','banana republic','j crew',
          'ann taylor','talbots','eileen fisher','free people','lululemon',
          'victoria secret','champion','hanes','north face','patagonia',
          'columbia','under armour','donna karan','dkny','anne klein',
          'amazon essentials','croft barrow','sonoma','a new day',
          'universal thread','denim co','susan graver','isaac mizrahi',
          'gloria vanderbilt','wrangler','lee','nydj','chico','soma','spanx']

def brand_type(name):
    if pd.isna(name): return 'Generic'
    n = str(name).lower()
    return 'Branded' if any(b in n for b in brands) else 'Generic'

df2['brand_type'] = df2['prod_name'].apply(brand_type)
print("\nBrand type counts:")
print(df2['brand_type'].value_counts())

# ─────────────────────────────────────────────
# STEP 6: RUN ALL ANALYSES
# ─────────────────────────────────────────────

# A: Household summary
hh = df2.groupby(['machine_id','customer_type']).agg(
    annual_transactions   = ('prod_totprice','count'),
    annual_spend          = ('prod_totprice','sum'),
    avg_price_per_txn     = ('prod_totprice','mean'),
    clothing_types_bought = ('clothing_type','nunique'),
    domains_used          = ('domain_name','nunique')
).reset_index()

A = hh.groupby('customer_type').agg(
    Households            = ('machine_id','count'),
    Avg_Transactions_Year = ('annual_transactions','mean'),
    Median_Transactions   = ('annual_transactions','median'),
    Avg_Annual_Spend      = ('annual_spend','mean'),
    Avg_Price_Per_Txn     = ('avg_price_per_txn','mean'),
    Avg_Clothing_Types    = ('clothing_types_bought','mean')
).round(2).reset_index()

# B: Monthly spending
B = df2.groupby(['customer_type','month','month_name']).agg(
    Total_Spend  = ('prod_totprice','sum'),
    Avg_Price    = ('prod_totprice','mean'),
    Transactions = ('prod_totprice','count')
).round(2).reset_index().sort_values(['customer_type','month'])

# C: Clothing type
C_raw = df2.groupby(['customer_type','clothing_type']).agg(
    Transactions = ('prod_totprice','count'),
    Avg_Price    = ('prod_totprice','mean'),
    Total_Spend  = ('prod_totprice','sum')
).round(2).reset_index()
C_raw['Txn_Share_Pct'] = (C_raw['Transactions'] /
    C_raw.groupby('customer_type')['Transactions'].transform('sum') * 100).round(1)

# D: Channel type
D_raw = df2.groupby(['customer_type','channel_type']).agg(
    Transactions = ('prod_totprice','count'),
    Avg_Price    = ('prod_totprice','mean')
).round(2).reset_index()
D_raw['Txn_Share_Pct'] = (D_raw['Transactions'] /
    D_raw.groupby('customer_type')['Transactions'].transform('sum') * 100).round(1)

# E: Branded vs Generic
E_raw = df2.groupby(['customer_type','brand_type']).agg(
    Transactions = ('prod_totprice','count'),
    Avg_Price    = ('prod_totprice','mean')
).round(2).reset_index()
E_raw['Txn_Share_Pct'] = (E_raw['Transactions'] /
    E_raw.groupby('customer_type')['Transactions'].transform('sum') * 100).round(1)

# F: Domain breakdown
F_raw = df2.groupby(['domain_name','customer_type']).agg(
    Transactions = ('prod_totprice','count'),
    Avg_Price    = ('prod_totprice','mean'),
    Total_Spend  = ('prod_totprice','sum')
).round(2).reset_index()

print("\nAll analyses complete. Building Excel file...")

# ─────────────────────────────────────────────
# STEP 7: BUILD EXCEL OUTPUT
# ─────────────────────────────────────────────
NAVY  = "1E2761"
BERRY = "6D2E46"
WHITE = "FFFFFF"
LIGHT = "EEF2F8"
ROSE  = "A26769"

def style_header(cell):
    cell.font      = Font(name='Arial', bold=True, color=WHITE, size=10)
    cell.fill      = PatternFill('solid', fgColor=NAVY)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    thin = Side(style='thin', color='CCCCCC')
    cell.border    = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_data(cell, bg=WHITE, align='center'):
    cell.font      = Font(name='Arial', color='1A1A1A', size=10)
    cell.fill      = PatternFill('solid', fgColor=bg)
    cell.alignment = Alignment(horizontal=align, vertical='center')
    thin = Side(style='thin', color='DDDDDD')
    cell.border    = Border(left=thin, right=thin, top=thin, bottom=thin)

def write_table(ws, df_in, start_row, title, col_widths=None):
    ws.merge_cells(start_row=start_row, start_column=1,
                   end_row=start_row, end_column=len(df_in.columns))
    tc = ws.cell(row=start_row, column=1, value=title)
    tc.font      = Font(name='Arial', bold=True, color=WHITE, size=11)
    tc.fill      = PatternFill('solid', fgColor=BERRY)
    tc.alignment = Alignment(horizontal='left', vertical='center', indent=1)
    ws.row_dimensions[start_row].height = 20

    hr = start_row + 1
    for ci, col in enumerate(df_in.columns, 1):
        style_header(ws.cell(row=hr, column=ci, value=col))
    ws.row_dimensions[hr].height = 18

    for ri, row_data in enumerate(df_in.itertuples(index=False), 1):
        r = hr + ri
        bg = LIGHT if ri % 2 == 0 else WHITE
        ws.row_dimensions[r].height = 16
        for ci, val in enumerate(row_data, 1):
            c = ws.cell(row=r, column=ci, value=val)
            style_data(c, bg=bg, align='left' if ci == 1 else 'center')

    if col_widths:
        for ci, w in enumerate(col_widths, 1):
            ws.column_dimensions[get_column_letter(ci)].width = w

    return hr + len(df_in) + 2

wb = Workbook()

# Sheet 1: Household Summary
ws1 = wb.active
ws1.title = "1. HH Summary"
ws1.sheet_view.showGridLines = False
ws1['A1'] = "Loyal vs Multi-Platform: Household Behaviour Summary"
ws1['A1'].font = Font(name='Arial', bold=True, size=14, color=NAVY)
ws1['A2'] = "comScore 2019 | Top 10 Women's Apparel Domains | prod_totprice basis"
ws1['A2'].font = Font(name='Arial', italic=True, size=10, color=ROSE)
write_table(ws1, A, 4,
    "Household-Level Behaviour: Loyal vs Multi-Platform",
    col_widths=[20,12,22,20,18,20,20])

# Sheet 2: Monthly Spending
ws2 = wb.create_sheet("2. Monthly Spending")
ws2.sheet_view.showGridLines = False
loyal_m = B[B['customer_type']=='Loyal'][['month_name','Total_Spend','Avg_Price','Transactions']].copy()
loyal_m.columns = ['Month','Total Spend ($)','Avg Price ($)','Transactions']
multi_m = B[B['customer_type']=='Multi-Platform'][['month_name','Total_Spend','Avg_Price','Transactions']].copy()
multi_m.columns = ['Month','Total Spend ($)','Avg Price ($)','Transactions']
row = 4
row = write_table(ws2, loyal_m, row, "Monthly Spending — LOYAL Shoppers",
    col_widths=[14,16,14,14])
write_table(ws2, multi_m, row, "Monthly Spending — MULTI-PLATFORM Shoppers",
    col_widths=[14,16,14,14])

# Sheet 3: Clothing Types
ws3 = wb.create_sheet("3. Clothing Types")
ws3.sheet_view.showGridLines = False
loyal_c = C_raw[C_raw['customer_type']=='Loyal'][
    ['clothing_type','Transactions','Avg_Price','Total_Spend','Txn_Share_Pct']].copy()
loyal_c.columns = ['Clothing Type','Transactions','Avg Price ($)','Total Spend ($)','Share (%)']
multi_c = C_raw[C_raw['customer_type']=='Multi-Platform'][
    ['clothing_type','Transactions','Avg_Price','Total_Spend','Txn_Share_Pct']].copy()
multi_c.columns = ['Clothing Type','Transactions','Avg Price ($)','Total Spend ($)','Share (%)']
row = 4
row = write_table(ws3, loyal_c.sort_values('Transactions', ascending=False), row,
    "Clothing Type — LOYAL Shoppers", col_widths=[18,14,14,14,12])
write_table(ws3, multi_c.sort_values('Transactions', ascending=False), row,
    "Clothing Type — MULTI-PLATFORM Shoppers", col_widths=[18,14,14,14,12])

# Sheet 4: Channel & Brand
ws4 = wb.create_sheet("4. Channel & Brand")
ws4.sheet_view.showGridLines = False
loyal_ch = D_raw[D_raw['customer_type']=='Loyal'][
    ['channel_type','Transactions','Avg_Price','Txn_Share_Pct']].copy()
loyal_ch.columns = ['Channel Type','Transactions','Avg Price ($)','Share (%)']
multi_ch = D_raw[D_raw['customer_type']=='Multi-Platform'][
    ['channel_type','Transactions','Avg_Price','Txn_Share_Pct']].copy()
multi_ch.columns = ['Channel Type','Transactions','Avg Price ($)','Share (%)']
loyal_br = E_raw[E_raw['customer_type']=='Loyal'][
    ['brand_type','Transactions','Avg_Price','Txn_Share_Pct']].copy()
loyal_br.columns = ['Brand Type','Transactions','Avg Price ($)','Share (%)']
multi_br = E_raw[E_raw['customer_type']=='Multi-Platform'][
    ['brand_type','Transactions','Avg_Price','Txn_Share_Pct']].copy()
multi_br.columns = ['Brand Type','Transactions','Avg Price ($)','Share (%)']
row = 4
row = write_table(ws4, loyal_ch,  row, "Channel Type — LOYAL Shoppers",     col_widths=[25,14,14,12])
row = write_table(ws4, multi_ch,  row, "Channel Type — MULTI-PLATFORM",     col_widths=[25,14,14,12])
row = write_table(ws4, loyal_br,  row, "Brand Type — LOYAL Shoppers",       col_widths=[14,14,14,12])
write_table(ws4,      multi_br,  row, "Brand Type — MULTI-PLATFORM",       col_widths=[14,14,14,12])

# Sheet 5: Domain Breakdown
ws5 = wb.create_sheet("5. Domain Breakdown")
ws5.sheet_view.showGridLines = False
F_pivot = F_raw.pivot_table(
    index='domain_name', columns='customer_type',
    values=['Transactions','Avg_Price','Total_Spend']
).round(2)
F_pivot.columns = [f"{b} - {a}" for a, b in F_pivot.columns]
F_pivot = F_pivot.reset_index()
write_table(ws5, F_pivot, 4,
    "Domain-Level: Loyal vs Multi-Platform (prod_totprice basis)",
    col_widths=[18,16,16,16,16,16,16])

# Sheet 6: Key Findings
ws6 = wb.create_sheet("6. Key Findings")
ws6.sheet_view.showGridLines = False
ws6['A1'] = "Key Findings — Loyal vs Multi-Platform Analysis"
ws6['A1'].font = Font(name='Arial', bold=True, size=14, color=NAVY)

findings = [
    ("FINDING", "RESULT", "WHAT IT MEANS", True),
    ("Annual spend per household", "Loyal: $62.95 | Multi: $212.83",
     "Multi-platform shoppers spend 3.4x more per year. Higher-value customers annually.", False),
    ("Transaction frequency", "Loyal: 2.52/year | Multi: 8.69/year",
     "Multi-platform shoppers visit online stores 3.5x more often.", False),
    ("Clothing type breadth", "Loyal: 1.53 types | Multi: 3.38 types",
     "Multi-platform shoppers explore more than double the clothing categories.", False),
    ("Channel preference", "Loyal: 67% single-channel | Multi: 59% dual-channel",
     "Loyal shoppers prefer online-only platforms. Multi-platform shoppers prefer retailers with physical stores.", False),
    ("Branded vs Generic", "Loyal: 35.4% branded | Multi: 36.6% branded",
     "No meaningful difference between groups.", False),
    ("Peak month", "Both groups peak in December",
     "Holiday effect is consistent across both customer types.", False),
    ("Kohl's and Gap attract cross-shoppers",
     "Kohl's: 2,059 multi vs 1,368 loyal",
     "These domains serve as secondary destinations for multi-platform shoppers.", False),
]

for ci, w in enumerate([28,32,50], 1):
    ws6.column_dimensions[get_column_letter(ci)].width = w

for ri, (f, r, m, is_hdr) in enumerate(findings, 3):
    ws6.row_dimensions[ri].height = 36
    bg = NAVY if is_hdr else (LIGHT if ri % 2 == 0 else WHITE)
    fg = WHITE if is_hdr else '1A1A1A'
    for ci, val in enumerate([f, r, m], 1):
        c = ws6.cell(row=ri, column=ci, value=val)
        c.font      = Font(name='Arial', bold=is_hdr, color=fg, size=10)
        c.fill      = PatternFill('solid', fgColor=bg)
        c.alignment = Alignment(horizontal='left', vertical='center',
                                wrap_text=True, indent=1)
        thin = Side(style='thin', color='CCCCCC')
        c.border    = Border(left=thin, right=thin, top=thin, bottom=thin)

wb.save('Loyalty_Full_Analysis.xlsx')
print("Done. Downloading file...")

from google.colab import files
files.download('Loyalty_Full_Analysis.xlsx')

Working sample: 25,891 transactions

Households by type:
customer_type
Loyal             6009
Multi-Platform    1238
Name: machine_id, dtype: int64

Clothing type counts:
clothing_type
Other         8428
Top           6784
Dress         2687
Pants         2399
Sweater       1544
Jeans         1264
Jacket         869
Swimwear       547
Activewear     508
Shoes          335
Skirt          302
Coat           224
Name: count, dtype: int64

Brand type counts:
brand_type
Generic    17511
Branded     8380
Name: count, dtype: int64

All analyses complete. Building Excel file...
Done. Downloading file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Incremental spend analysis
df_inc = df[df['prod_totprice'] > 0].copy()
df_inc['incidental_spend'] = df_inc['basket_tot'] - df_inc['prod_totprice']
df_inc['incidental_spend'] = df_inc['incidental_spend'].clip(lower=0)

# Summary
print("=== INCIDENTAL SPEND ANALYSIS ===")
print(f"Total transactions: {len(df_inc):,}")
print(f"Transactions with zero incidental spend: {(df_inc['incidental_spend']==0).sum():,}")
print(f"Transactions with positive incidental spend: {(df_inc['incidental_spend']>0).sum():,}")
print(f"\nAvg incidental spend (all transactions): ${df_inc['incidental_spend'].mean():.2f}")
print(f"Avg incidental spend (where >0 only): ${df_inc[df_inc['incidental_spend']>0]['incidental_spend'].mean():.2f}")

# Task 2: where incidental > clothing spend
df_inc['incidental_exceeds_clothing'] = df_inc['incidental_spend'] > df_inc['prod_totprice']
print(f"\nTransactions where incidental spend > clothing spend: {df_inc['incidental_exceeds_clothing'].sum():,}")
print(f"As % of all transactions: {df_inc['incidental_exceeds_clothing'].mean():.1%}")

# By domain
print("\n=== BY DOMAIN ===")
domain_inc = df_inc.groupby('domain_name').agg(
    Transactions = ('prod_totprice','count'),
    Avg_Clothing_Price = ('prod_totprice','mean'),
    Avg_Basket = ('basket_tot','mean'),
    Avg_Incidental = ('incidental_spend','mean'),
    Pct_Incidental_Exceeds = ('incidental_exceeds_clothing','mean')
).round(2)
domain_inc['Pct_Incidental_Exceeds'] = (domain_inc['Pct_Incidental_Exceeds']*100).round(1)
domain_inc = domain_inc.sort_values('Avg_Incidental', ascending=False)
print(domain_inc.to_string())

=== INCIDENTAL SPEND ANALYSIS ===
Total transactions: 25,891
Transactions with zero incidental spend: 3,937
Transactions with positive incidental spend: 21,954

Avg incidental spend (all transactions): $58.48
Avg incidental spend (where >0 only): $68.97

Transactions where incidental spend > clothing spend: 15,225
As % of all transactions: 58.8%

=== BY DOMAIN ===
                 Transactions  Avg_Clothing_Price  Avg_Basket  Avg_Incidental  Pct_Incidental_Exceeds
domain_name                                                                                          
blair.com                1167               20.89      124.64          103.90                    85.0
kohls.com                3427               22.15      109.75           88.37                    82.0
gap.com                  1416               24.53      106.79           83.71                    79.0
qvc.com                   787               40.71      115.35           74.86                    54.0
walmart.com          

In [11]:
# Inter-purchase time analysis
df['event_date'] = pd.to_datetime(df['event_date'])
df_dates = df[df['prod_totprice'] > 0].copy()

# Sort by machine_id and date
df_dates = df_dates.sort_values(['machine_id','event_date'])

# Calculate days between purchases per household
df_dates['prev_date'] = df_dates.groupby('machine_id')['event_date'].shift(1)
df_dates['days_between'] = (df_dates['event_date'] - df_dates['prev_date']).dt.days

# Only keep rows where there was a previous purchase
gap_df = df_dates[df_dates['days_between'].notna()].copy()

print("=== INTER-PURCHASE TIME ===")
print(f"Households with 2+ purchases: {gap_df['machine_id'].nunique():,}")
print(f"\nOverall inter-purchase time (days):")
print(gap_df['days_between'].describe().round(1))

# By customer type - need to merge customer_type
top10 = ['amazon.com','kohls.com','walmart.com','ebay.com','gap.com',
         'macys.com','blair.com','womanwithin.com','target.com','qvc.com']
df_top = df[df['domain_name'].isin(top10) & (df['prod_totprice']>0)].copy()
domains_per_hh = df_top.groupby('machine_id')['domain_name'].nunique().reset_index(name='n_domains')
domains_per_hh['customer_type'] = domains_per_hh['n_domains'].apply(
    lambda x: 'Loyal' if x == 1 else 'Multi-Platform')
gap_df = gap_df.merge(domains_per_hh[['machine_id','customer_type']], on='machine_id', how='left')

print("\nInter-purchase time by customer type (days):")
print(gap_df.groupby('customer_type')['days_between'].agg(
    ['mean','median','min','max']).round(1))

# Quantity analysis
print("\n=== PRODUCT QUANTITY ANALYSIS ===")
qty_df = df_top.merge(domains_per_hh[['machine_id','customer_type']], on='machine_id')
print("Quantity distribution overall:")
print(qty_df['prod_qty'].describe().round(2))

print("\nAvg quantity by customer type:")
print(qty_df.groupby('customer_type')['prod_qty'].agg(
    ['mean','median','sum','count']).round(2))

print("\nQuantity distribution (value counts):")
print(qty_df['prod_qty'].value_counts().head(10).sort_index())

=== INTER-PURCHASE TIME ===
Households with 2+ purchases: 4,039

Overall inter-purchase time (days):
count    18644.0
mean        21.2
std         49.7
min          0.0
25%          0.0
50%          0.0
75%         14.0
max        357.0
Name: days_between, dtype: float64

Inter-purchase time by customer type (days):
                mean  median  min    max
customer_type                           
Loyal           19.5     0.0  0.0  357.0
Multi-Platform  22.7     0.0  0.0  341.0

=== PRODUCT QUANTITY ANALYSIS ===
Quantity distribution overall:
count    25891.00
mean         1.06
std          0.40
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         20.00
Name: prod_qty, dtype: float64

Avg quantity by customer type:
                mean  median    sum  count
customer_type                             
Loyal           1.07     1.0  16175  15137
Multi-Platform  1.05     1.0  11249  10754

Quantity distribution (value counts):
prod_qty
1     24874
2       774
3

In [12]:
print(df.shape)

(26098, 56)


In [13]:
import statsmodels.api as sm
import numpy as np

# ─────────────────────────────────────────────
# STEP 1: PREPARE DATA FOR REGRESSION
# ─────────────────────────────────────────────

# Filter out zero prod_totprice
df_reg = df[df['prod_totprice'] > 0].copy()
print(f"Sample after removing zero prod_totprice: {len(df_reg):,}")

# Winsorize prod_totprice at 99th percentile
p99 = df_reg['prod_totprice'].quantile(0.99)
df_reg['prod_totprice_wins'] = df_reg['prod_totprice'].clip(upper=p99)
print(f"99th percentile cap: ${p99:.2f}")

# Log dependent variable
df_reg['log_prod_price'] = np.log(df_reg['prod_totprice_wins'] + 1)

# ─────────────────────────────────────────────
# STEP 2: CREATE DOMAIN DUMMIES (amazon = reference)
# ─────────────────────────────────────────────
domains = ['kohls.com','walmart.com','ebay.com','gap.com',
           'macys.com','blair.com','womanwithin.com','target.com','qvc.com']
for d in domains:
    col = 'D_' + d.replace('.','_')
    df_reg[col] = (df_reg['domain_name'] == d).astype(int)

# Quarter dummies (Q1 = reference)
for q in [2, 3, 4]:
    df_reg[f'Q{q}'] = (df_reg['Quarter'] == q).astype(int)

# Region dummies (Northeast = reference)
for r in [2, 3, 4]:
    df_reg[f'Region_{r}'] = (df_reg['census_region Code'] == r).astype(int)

# ─────────────────────────────────────────────
# STEP 3: DEFINE ALL VARIABLES
# ─────────────────────────────────────────────
domain_vars  = ['D_' + d.replace('.','_') for d in domains]
quarter_vars = ['Q2','Q3','Q4']
region_vars  = ['Region_2','Region_3','Region_4']
hh_size_vars = ['HH_Size_2','HH_Size_3','HH_Size_4','HH_Size_5','HH_Size_6plus']
income_vars  = ['Income_25_39k','Income_40_59k','Income_60_74k','Income_75_99k',
                'Income_100_149k','Income_150_199k','Income_200k_plus']
age_vars     = ['Gen_Z_YoungMillennial','Millennial','GenX']
edu_vars     = ['Edu_Some_College','Edu_Associate','Edu_Bachelors',
                'Edu_Graduate','Edu_Missing']
other_vars   = ['children Code','country_of_origin Code']

all_vars = (domain_vars + quarter_vars + region_vars + hh_size_vars +
            income_vars + age_vars + edu_vars + other_vars)

# Check all variables exist
missing = [v for v in all_vars if v not in df_reg.columns]
print(f"\nMissing variables: {missing}")

# ─────────────────────────────────────────────
# STEP 4: BUILD X AND Y
# ─────────────────────────────────────────────
X = df_reg[all_vars].copy()
X = sm.add_constant(X)
y = df_reg['log_prod_price']

# Drop rows with missing values
clean = X.notna().all(axis=1) & y.notna()
X = X[clean]
y = y[clean]
groups = df_reg[clean]['machine_id']

print(f"\nFinal sample size: {len(y):,}")
print(f"Number of predictors: {X.shape[1]}")

# ─────────────────────────────────────────────
# STEP 5: RUN OLS WITH CLUSTERED STANDARD ERRORS
# ─────────────────────────────────────────────
model  = sm.OLS(y, X)
result = model.fit(cov_type='cluster', cov_kwds={'groups': groups})

print(result.summary())
print(f"\nSample size:         {int(result.nobs):,}")
print(f"R-squared:           {result.rsquared:.4f}")
print(f"Adjusted R-squared:  {result.rsquared_adj:.4f}")
print(f"99th pct winsor cap: ${p99:.2f}")

# ─────────────────────────────────────────────
# STEP 6: SAVE RESULTS TO CSV
# ─────────────────────────────────────────────
import pandas as pd

def stars(p):
    if p < 0.01:  return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    else:          return ''

results_df = pd.DataFrame({
    'Variable':     result.params.index,
    'Coefficient':  result.params.values.round(4),
    'Std_Error':    result.bse.values.round(4),
    'z_stat':       result.tvalues.values.round(3),
    'p_value':      result.pvalues.values.round(4),
    'Significance': [stars(p) for p in result.pvalues]
})

results_df.to_csv('model2_prod_totprice_results.csv', index=False)
print("\nResults saved to model2_prod_totprice_results.csv")

from google.colab import files
files.download('model2_prod_totprice_results.csv')

Sample after removing zero prod_totprice: 25,891
99th percentile cap: $111.99

Missing variables: []

Final sample size: 25,891
Number of predictors: 38
                            OLS Regression Results                            
Dep. Variable:         log_prod_price   R-squared:                       0.151
Model:                            OLS   Adj. R-squared:                  0.150
Method:                 Least Squares   F-statistic:                     48.15
Date:                Sun, 03 May 2026   Prob (F-statistic):          7.86e-312
Time:                        19:04:03   Log-Likelihood:                -22822.
No. Observations:               25891   AIC:                         4.572e+04
Df Residuals:                   25853   BIC:                         4.603e+04
Df Model:                          37                                         
Covariance Type:              cluster                                         
                             coef    std err          z  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Load and prepare
top10 = ['amazon.com','kohls.com','walmart.com','ebay.com','gap.com',
         'macys.com','blair.com','womanwithin.com','target.com','qvc.com']
df_m = df[df['domain_name'].isin(top10) & (df['prod_totprice']>0)].copy()

# Create switching dependent variable at household level
hh = df_m.groupby('machine_id').agg(
    n_domains       = ('domain_name','nunique'),
    income_code     = ('household_income Code','first'),
    age_code        = ('hoh_oldest_age Code','first'),
    edu_code        = ('hoh_most_education Code','first'),
    hh_size         = ('household_size Code','first'),
    children        = ('children Code','first'),
    region          = ('census_region Code','first'),
    Age_Group       = ('Age_Group','first')
).reset_index()

hh['switcher'] = (hh['n_domains'] > 1).astype(int)

# Standardize income (z-score)
hh['income_std'] = (hh['income_code'] - hh['income_code'].mean()) / hh['income_code'].std()

# Age dummy: Young = Gen Z + Millennial = 1, else 0
hh['young_shopper'] = hh['Age_Group'].isin(
    ['Gen Z / Young Millennial','Millennial']).astype(int)

# Education dummies (HS diploma reference)
for code, name in [(2,'Some_College'),(3,'Associate'),(4,'Bachelors'),
                   (5,'Graduate'),(99,'Edu_Missing')]:
    hh[f'Edu_{name}'] = (hh['edu_code'] == code).astype(int)

# Household size dummies (size 1 reference)
for s in [2,3,4,5]:
    hh[f'HH_Size_{s}'] = (hh['hh_size'] == s).astype(int)

# Region dummies (Northeast reference)
for r in [2,3,4]:
    hh[f'Region_{r}'] = (hh['region'] == r).astype(int)

# Build X and y
feature_cols = (['income_std','young_shopper','children'] +
                [f'Edu_{n}' for n in ['Some_College','Associate','Bachelors','Graduate','Edu_Missing']] +
                [f'HH_Size_{s}' for s in [2,3,4,5]] +
                [f'Region_{r}' for r in [2,3,4]])

X = hh[feature_cols].copy()
X = sm.add_constant(X)
y = hh['switcher']

clean = X.notna().all(axis=1) & y.notna()
X = X[clean]
y = y[clean]

# Run logistic regression
logit_model = sm.Logit(y, X)
logit_result = logit_model.fit()

print(logit_result.summary())
print(f"\nTotal households: {len(y):,}")
print(f"Switchers: {y.sum():,} ({y.mean():.1%})")
print(f"Loyal: {(1-y).sum():,} ({(1-y).mean():.1%})")

Optimization terminated successfully.
         Current function value: 0.446768
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               switcher   No. Observations:                 7247
Model:                          Logit   Df Residuals:                     7231
Method:                           MLE   Df Model:                           15
Date:                Sun, 03 May 2026   Pseudo R-squ.:                 0.02281
Time:                        19:04:03   Log-Likelihood:                -3237.7
converged:                       True   LL-Null:                       -3313.3
Covariance Type:            nonrobust   LLR p-value:                 1.402e-24
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -1.6371      0.295     -5.547      0.000      -2.216      -1.059
income_std 

In [15]:
import pandas as pd

def stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    else: return ''

logit_df = pd.DataFrame({
    'Variable': logit_result.params.index,
    'Coefficient': logit_result.params.values.round(4),
    'Std_Error': logit_result.bse.values.round(4),
    'z_stat': logit_result.tvalues.values.round(3),
    'p_value': logit_result.pvalues.values.round(4),
    'Significance': [stars(p) for p in logit_result.pvalues]
})

logit_df.to_csv('model_B_switching.csv', index=False)
print(logit_df.to_string(index=False))

from google.colab import files
files.download('model_B_switching.csv')

        Variable  Coefficient  Std_Error  z_stat  p_value Significance
           const      -1.6371     0.2951  -5.547   0.0000          ***
      income_std       0.1224     0.0340   3.604   0.0003          ***
   young_shopper      -0.3327     0.0798  -4.170   0.0000          ***
        children       0.2055     0.0830   2.475   0.0133           **
Edu_Some_College       0.4439     0.2828   1.570   0.1165             
   Edu_Associate       0.4522     0.2828   1.599   0.1097             
   Edu_Bachelors       0.4312     0.2853   1.512   0.1306             
    Edu_Graduate       0.5320     0.3119   1.706   0.0881            *
 Edu_Edu_Missing      -0.1416     0.2879  -0.492   0.6229             
       HH_Size_2      -0.1328     0.1151  -1.154   0.2486             
       HH_Size_3      -0.0693     0.1294  -0.536   0.5923             
       HH_Size_4       0.0167     0.1398   0.119   0.9052             
       HH_Size_5      -0.1136     0.1402  -0.810   0.4181             
      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print(df.columns.tolist())

['machine_id', 'site_session_id', 'prod_category_id', 'prod_name', 'domain_id', 'prod_qty', 'prod_totprice', 'basket_tot', 'event_date', 'event_time', 'hoh_most_education Code', 'hoh_most_education', 'census_region Code', 'census_region', 'household_size Code', 'household_size', 'hoh_oldest_age Code', 'hoh_oldest_age', 'household_income Code', 'household_income', 'children Code', 'children', 'racial_background Code', 'racial_background', 'connection_speed Code', 'connection_speed Code.1', 'country_of_origin Code', 'country_of_origin', 'zip_code', 'City Name', 'State', 'domain_name', 'Quarter', 'Income_Label', 'Age_Group', 'HH_Size_2', 'HH_Size_3', 'HH_Size_4', 'HH_Size_5', 'HH_Size_6plus', 'Income_25_39k', 'Income_40_59k', 'Income_60_74k', 'Income_75_99k', 'Income_100_149k', 'Income_150_199k', 'Income_200k_plus', 'Gen_Z_YoungMillennial', 'Millennial', 'GenX', 'Edu_HS_Diploma', 'Edu_Some_College', 'Edu_Associate', 'Edu_Bachelors', 'Edu_Graduate', 'Edu_Missing']


In [17]:
import numpy as np
import statsmodels.api as sm

# Create quarter dummies from Quarter column (Q1 is reference)
df['Q2'] = (df['Quarter'] == 2).astype(int)
df['Q3'] = (df['Quarter'] == 3).astype(int)
df['Q4'] = (df['Quarter'] == 4).astype(int)

# Create children binary (1 = children present, 0 = none)
df['Children'] = (df['children'] == 'Yes').astype(int)

# Create log dependent variable
df['log_prod_totprice'] = np.log(df['prod_totprice'] + 1)

controls = [
    'Q2', 'Q3', 'Q4',
    'HH_Size_2', 'HH_Size_3', 'HH_Size_4', 'HH_Size_5', 'HH_Size_6plus',
    'Income_25_39k', 'Income_40_59k', 'Income_60_74k', 'Income_75_99k',
    'Income_100_149k', 'Income_150_199k', 'Income_200k_plus',
    'Edu_Some_College', 'Edu_Associate', 'Edu_Bachelors',
    'Edu_Graduate', 'Edu_Missing',
    'Children'
]

X = df[controls]
X = sm.add_constant(X)
y = df['log_prod_totprice']

model = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df['machine_id']})
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:      log_prod_totprice   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     5.939
Date:                Sun, 03 May 2026   Prob (F-statistic):           1.41e-16
Time:                        19:04:03   Log-Likelihood:                -27587.
No. Observations:               26098   AIC:                         5.522e+04
Df Residuals:                   26076   BIC:                         5.540e+04
Df Model:                          21                                         
Covariance Type:              cluster                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                2.9223      0.061  

In [18]:
from google.colab import files
import pandas as pd

results_df = pd.DataFrame({
    'Variable': model.params.index,
    'Coefficient': model.params.values.round(4),
    'Std_Error': model.bse.values.round(4),
    'z_stat': model.tvalues.values.round(3),
    'p_value': model.pvalues.values.round(4)
})

results_df.to_csv('model_consumer_only.csv', index=False)
files.download('model_consumer_only.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import numpy as np
import statsmodels.api as sm

# Create interaction terms
df['HHSize_x_Income_25_39k'] = df['HH_Size_2'] * df['Income_25_39k']
# ... repeat for all meaningful combinations

# Example: education x income interactions
edu_vars = ['Edu_Some_College', 'Edu_Bachelors', 'Edu_Graduate']
income_vars = ['Income_60_74k', 'Income_100_149k', 'Income_200k_plus']

for edu in edu_vars:
    for inc in income_vars:
        df[f'{edu}_x_{inc}'] = df[edu] * df[inc]

# Same for household size x education
hhsize_vars = ['HH_Size_2', 'HH_Size_3', 'HH_Size_4', 'HH_Size_5', 'HH_Size_6plus']
for hh in hhsize_vars:
    for edu in edu_vars:
        df[f'{hh}_x_{edu}'] = df[hh] * df[edu]

In [20]:
# Build full interaction controls list
interaction_cols = (
    [f'{edu}_x_{inc}' for edu in edu_vars for inc in income_vars] +
    [f'{hh}_x_{edu}' for hh in hhsize_vars for edu in edu_vars]
)

base_controls = [
    'Q2', 'Q3', 'Q4',
    'HH_Size_2', 'HH_Size_3', 'HH_Size_4', 'HH_Size_5', 'HH_Size_6plus',
    'Income_25_39k', 'Income_40_59k', 'Income_60_74k', 'Income_75_99k',
    'Income_100_149k', 'Income_150_199k', 'Income_200k_plus',
    'Edu_Some_College', 'Edu_Associate', 'Edu_Bachelors',
    'Edu_Graduate', 'Edu_Missing',
    'Children'
]

all_controls = base_controls + interaction_cols

X = df[all_controls]
X = sm.add_constant(X)
y = df['log_prod_totprice']

model_interaction = sm.OLS(y, X).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['machine_id']}
)
print(model_interaction.summary())

                            OLS Regression Results                            
Dep. Variable:      log_prod_totprice   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     3.235
Date:                Sun, 03 May 2026   Prob (F-statistic):           1.09e-11
Time:                        19:04:03   Log-Likelihood:                -27555.
No. Observations:               26098   AIC:                         5.520e+04
Df Residuals:                   26055   BIC:                         5.555e+04
Df Model:                          42                                         
Covariance Type:              cluster                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 45, but rank is 42
  warnings.warn('covariance of constraints does not have full '


In [21]:
# Remove HH_Size_6plus interactions - empty cells cause singularity
interaction_cols_clean = (
    [f'{edu}_x_{inc}' for edu in edu_vars for inc in income_vars] +
    [f'{hh}_x_{edu}' for hh in
     ['HH_Size_2','HH_Size_3','HH_Size_4','HH_Size_5']  # exclude 6plus
     for edu in edu_vars]
)

all_controls_clean = base_controls + interaction_cols_clean

X = df[all_controls_clean]
X = sm.add_constant(X)
y = df['log_prod_totprice']

model_interaction2 = sm.OLS(y, X).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['machine_id']}
)
print(model_interaction2.summary())

                            OLS Regression Results                            
Dep. Variable:      log_prod_totprice   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     3.235
Date:                Sun, 03 May 2026   Prob (F-statistic):           1.08e-11
Time:                        19:04:03   Log-Likelihood:                -27555.
No. Observations:               26098   AIC:                         5.520e+04
Df Residuals:                   26055   BIC:                         5.555e+04
Df Model:                          42                                         
Covariance Type:              cluster                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

In [22]:
from google.colab import files
import pandas as pd

results_df = pd.DataFrame({
    'Variable': model_interaction2.params.index,
    'Coefficient': model_interaction2.params.values.round(4),
    'Std_Error': model_interaction2.bse.values.round(4),
    'z_stat': model_interaction2.tvalues.values.round(3),
    'p_value': model_interaction2.pvalues.values.round(4)
})

results_df.to_csv('model_interaction_results.csv', index=False)
files.download('model_interaction_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>